## Network Analysis with NetworkX

This notebook is based on: https://github.com/rossanoventurini/adsds/blob/main/Lab/Lecture_08/L08_Graphs_with_NetworkX_no_sols.ipynb

![](img/NetworkX.jpg)

[NetworkX](https://networkx.github.io/) is a Python package for the creation, manipulation, and study of the structure, dynamics, and functions of complex networks.

- Data structures for graphs, digraphs, and multigraphs
- Many standard graph algorithms
- Network structure and analysis measures
- Generators for classic graphs, random graphs, and synthetic networks
- Nodes can be "anything" (e.g., text, images, XML records)
- Edges can hold arbitrary data (e.g., weights, time-series)

---

### Get prepared

#### Installations

tbd

#### Imports

In [ ]:
import networkx as nx
print(nx.__version__)
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import numpy as np

---

### Functions to plot graphs 

In [ ]:
def plot_graph(G, size=5):
    plt.figure(figsize=(size, size))
    nx.draw(G, with_labels=True, node_color='skyblue', width=.3, font_size=8)
    plt.show()

def plot_graph_with_weights(G):
    pos = nx.planar_layout(G) # pos = nx.nx_agraph.graphviz_layout(G)
    nx.draw_networkx(G,pos)
    labels = nx.get_edge_attributes(G,'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
    plt.show()

---

### Basic graph construction

#### In advance

In Python, a node is a tuple of the form `(n, node_attribute_dict)`.

Similar are edges: `(n1, n2, edge_attribute_dict)`

#### Create a Graph

A graph is a __collection of nodes and edges__, i.e., G = (V, E).

Let's start by creating an empty undirected graph

In [ ]:
G = nx.Graph()

The graph object has some __properties__ and __methods__ giving data about the whole graph, see below.

#### Populate the graph.


 We can add one node at time using the `add-node` method or add any iterable collections (lists, strings etc..) with the `G.add_nodes_from` method. 

We are now using strings as nodes's names, but nodes can be any hashable object.

In [ ]:
G.add_node('a')
my_list = ['b', 'c', 'd']
G.add_nodes_from(my_list)

We can __inspect the nodes__ in a graph using *G.nodes()*.

In [ ]:
G.nodes()
list(G.nodes())

In [ ]:
plot_graph(G)

Now let's add edges to the graph. 

We can add one edge at time with *G.add_edge()*, specifying the nodes between whom we want the edge to stay. As for nodes, we can pass an iterable colletion to *G.add_edges_from()*.

In [ ]:
G.add_edges_from([('a', 'b'), ('a', 'c'), ('c', 'd'), ('b', 'd')])
G.edges()
list(G.edges())

If we add edges between nodes that do not exist, __theses nodes are automatically added__.

In [ ]:
G.add_edge("a", "e")
G.add_edge("a", "b")

In [ ]:
# Number of nodes
print("Number of Nodes: ", G.order()) # equivalent to len(G) and G.number_of_nodes()
# Number of edges
print("Number of Edges: ", G.number_of_edges())

In [ ]:
plot_graph(G, 12)

__Remove__ nodes using the *G.remove_node()* or the *G.remove_nodes_from()*

In [ ]:
G.remove_node('e')
plot_graph(G)

#### Node attributes

In [ ]:
for node in G.nodes:
    G.nodes[node]['color'] = 'white'

In [ ]:
list(G.nodes(data=True))

In [ ]:
G.nodes['a']['color']

#### Edge attributes

In [ ]:
G['a']['b']['color'] = 'green'
G['a']['c']['color'] = 'black'
list(G.edges(data=True))

---

### Neighbours and node degree

We can explore the **neighbours** and the **degree** of the nodes in a graph using the facilities *G.adj* and *G.degree*.

In [ ]:
print(G.adj)

Graph are __subscriptable__ objects and we can access neighbours directly with the [ ] operator.

In [ ]:
print(G.adj['a'])
print(G['a'])

We can use *G.adj* as a dictionary and __iterate__ over it.

In [ ]:
for node, attr in G.adj.items():
    # attr is a dictionary whose keys are node neighbours and whose value, in turn, is a dictionary 
    # reporting 'property:value' of the edge between node and key
    
    print(node, attr) 

The degree property reports the number of edges for each node.

In [ ]:
print(G.degree)

We can compute a degree histogram.

In [ ]:
hist = nx.degree_histogram(G)
hist

---

### Graph Operations

Networkx supports most common graph operations. Some examples are 
<ul>
<li>Union</li>
<li>Intersection</li>
<li>Complement</li>
</ul>

In [ ]:
# Create a first graph G
G = nx.Graph()
G.add_edges_from([("a", "b"), ("b", "c"), ("c", "d")])

# Create a second graph H
H = nx.Graph()
H.add_edges_from([("e", "f"), ("f", "g"), ("g", "h") ])

# Perform the union operation
I = nx.union(G, H)
plot_graph(I)

In [ ]:
# Now link the two connected components
I.add_edges_from([("a","e"), ("b", "g")])
plot_graph(I)

In [ ]:
# Create a third graph
L = nx.Graph()

# To perform intersection between two graphs, they must have the same nodes
# First we copy all the nodes in I into L
L.add_nodes_from(I.nodes())

# Then we add some edge in L
L.add_edges_from([("e", "f"), ("f", "g")])

M = nx.intersection(I, L)

# Only the shared edges between I, L wil survive this operation
plot_graph(M)

---

### Create graphs from Netflix Streaming dataset

#### Purpose

In this notebook, I'm going to construct three  graphs as follows:
1. I construct a bipartite graph `B` from the dataframe prepared above, where `B` comprises two node sets, namely the _actor name_ node set and _movie title_ node set. I call this graph the _**movie-actor graph**_.
1. I apply projection onto the actor nodes to create the graph `C` from the bipartite graph, which I call the _**co-actor graph**_.
1. I apply projection onto the title nodes to create the graph `S` from the bipartite graph, which I call the _**shared-actor graph**_.

![](img/three%20graphs.png)

#### Load data

In [ ]:
# Load dataset
df_clean = pd.read_pickle('./datasets/Netflix Streaming Data/Netflix Streaming Data-Cleaned.pkl')
print(df_clean.info())
df_clean.head()

---

#### Remove unnecessary rows: TV shows

In [ ]:
# Consider only rows of type = "Movie".
df_movie = df_clean[df_clean['type'] == 'Movie']
df_movie = df_movie.reset_index(drop=True)
df_movie.info()

---

#### Check whether `show_id` entries are unique.

For the moment, we have 5633 movies. For constructing the bipartite graph, we need unique identifiers for movies.

In [7]:
print(len(set(df_movie["show_id"])))
print(len(df_movie["show_id"].unique()))

5633
5633


---

#### Check whether `title` entries are unique.

In [8]:
print(len(set(df_movie["title"].str.strip())))
print(len(df_movie["title"].unique()))

5633
5633


---

#### Create a set of `actor_names` from the `cast` entries.

In [9]:
movie_titles = set(df_movie["title"].str.strip())
actor_names  = set()

for idx, row in df_movie.iterrows():
    actor_names.update([actor.strip() for actor in row['cast'].split(',')])

print(f"Total number of movie titles: {len(movie_titles)}")
print(f"Total number of actor names: {len(actor_names)}")

Total number of movie titles: 5633
Total number of actor names: 25948


---

#### Check whether there is an intersection between `movie_titles` and `actor_names`.

In [10]:
intersect = movie_titles.intersection(actor_names)
print(f"Movie titles that are also actor names: {intersect}")

Movie titles that are also actor names: {'Shiva', 'Tarzan', 'Sebastián Marcelo Wainraich', 'Jimi Hendrix', 'Amar', 'Secret', 'Max Rose', 'Solo', 'Game'}


---

#### Create bipartite graph `B`

In [ ]:
B = nx.Graph()

for idx, row in df_movie.iterrows():
    title = row['title']
    actors = [actor.strip() for actor in row['cast'].split(',')]
    for actor in actors:
        B.add_edge(title, actor)
        
for node in list(B.nodes(data=True))[210:230]:
    print(node)

('Scott Glenn', {})
('Tom Berenger', {})
('Harris Yulin', {})
('Raymond J. Barry', {})
('Cliff Curtis', {})
('Dr. Dre', {})
('Snoop Dogg', {})
('Macy Gray', {})
('Eva Mendes', {})
('InuYasha the Movie 2: The Castle Beyond the Looking Glass', {})
('Kappei Yamaguchi', {})
('Satsuki Yukino', {})
('Mieko Harada', {})
('Koji Tsujitani', {})
('Houko Kuwashima', {})
('Kumiko Watanabe', {})
('Noriko Hidaka', {})
('Kenichi Ogata', {})
('Toshiyuki Morikawa', {})
('Izumi Ogami', {})


In [6]:
for edge in list(B.edges(data=True))[210:230]:
    print(edge)

('Kajol', 'We Are Family', {})
('Kajol', 'Dilwale', {})
('Prabhu Deva', 'ABCD 2', {})
('Prabhu Deva', 'ABCD: Any Body Can Dance', {})
('Prabhu Deva', 'Abhinetri', {})
('Girish Karnad', 'Mugamoodi', {})
('Girish Karnad', 'Aashayein', {})
('Girish Karnad', 'Iqbal', {})
('Girish Karnad', 'Swami', {})
('Grown Ups', 'Adam Sandler', {})
('Grown Ups', 'Kevin James', {})
('Grown Ups', 'Chris Rock', {})
('Grown Ups', 'David Spade', {})
('Grown Ups', 'Rob Schneider', {})
('Grown Ups', 'Salma Hayek', {})
('Grown Ups', 'Maria Bello', {})
('Grown Ups', 'Maya Rudolph', {})
('Grown Ups', 'Colin Quinn', {})
('Grown Ups', 'Tim Meadows', {})
('Grown Ups', 'Joyce Van Patten', {})


---

#### Some statistics

Number of connected components

Degree histogram for `movie title` nodes